# Fleet and Equipment Analysis

## Goal

This notebook examines whether fleet and equipment characteristics (truck make, truck age,
trailer type) are associated with delivery performance, how fuel efficiency and equipment
utilization behave across the fleet, and how diesel prices developed over the three years.

1. What does the fleet look like, and which parts of it are actually used?
2. Do delay and duration differ by truck characteristics or trailer type?
3. Do individual trucks stand out, or is the spread consistent with chance?
4. How does fuel efficiency (MPG) vary across trucks, trailers, and trips?
5. How did diesel prices develop over time?
6. How do utilization, maintenance cost, and downtime develop over time?

## Data Basis

- `trucks`, `trailers`, `fuel_purchases`, `truck_utilization_metrics` - new for this notebook
- `loads`, `trips`, `delivery_events` - already loaded via the pipeline, providing
  `delivery_delay_hours`, `is_delayed_delivery`, `delivery_duration_hours`

*Note on the data: the dataset is described by its source as a realistic simulation.
Earlier notebooks found little variation across time, region, and equipment; this notebook
therefore places particular emphasis on testing whether an effect is real or consistent with
random variation, rather than on the size of the findings.*

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from delivery_pipeline import run_pipeline, PROCESSED_DATA_PATH_FLEET, RAW_DATA_DIR

clean_df = run_pipeline(output_path=PROCESSED_DATA_PATH_FLEET, include_routes=False, include_fleet=True)
clean_df = clean_df.drop(columns=["route_id"])
print(f"Loads used for analysis: {len(clean_df)}")

fuel_df = pd.read_csv(RAW_DATA_DIR / "fuel_purchases.csv")
util_df = pd.read_csv(RAW_DATA_DIR / "truck_utilization_metrics.csv")
trucks_df = pd.read_csv(RAW_DATA_DIR / "trucks.csv")
trips_df = pd.read_csv(RAW_DATA_DIR / "trips.csv")
trailers_df = pd.read_csv(RAW_DATA_DIR / "trailers.csv")
loads_df = pd.read_csv(RAW_DATA_DIR / "loads.csv")
delivery_events_df = pd.read_csv(RAW_DATA_DIR / "delivery_events.csv")

print(f"Fuel purchases: {len(fuel_df)}, Utilization records: {len(util_df)}")

sns.set_theme(style="darkgrid")



*Note: `trucks_df`, `trailers_df`, `loads_df`, and `delivery_events_df` are loaded
directly from the raw CSVs, in addition to `clean_df` from the pipeline. `fuel_df` and
`util_df` have a different grain (one row per fuel purchase / per truck-month) and are
therefore never merged into the load-level table. `trucks_df`/`trailers_df` are
reference tables, not on `clean_df`'s grain either. `loads_df`/`delivery_events_df`
are used specifically for the row-count comparison in section 2.4 — comparing against
`clean_df` would understate the counts, since the pipeline already dropped 486
inconsistent-timestamp loads and flattened the event-level table.*

## 2. Data Quality Checks
### 2.1 How many trips have missing truck, trailer, or driver references?

In [ ]:
display(clean_df[['truck_id','trailer_id','driver_id']].isna().mean())
display((clean_df.truck_id.isna() & clean_df.trailer_id.isna()).sum())

*Data Quality Note: `truck_id`/`trailer_id`/`driver_id` are missing in about 2% of
rows, consistent with the "intentional 2% null rate" the dataset description states.
`trucks` and `trips` are loaded directly from the raw CSVs (see section 1),
rather than via `clean_df.truck_id`. The pipeline drops 486 loads (0.57%) with
inconsistent timestamps; a truck that only appears in those dropped loads would
otherwise look unused even though it has trips. Loading `trips` raw avoids that
edge case. 36 loads are missing both `truck_id` and `trailer_id` simultaneously, roughly
what independent ~2% null rates would predict by chance.*

### 2.2 Which trucks never appear in trips, and why?

In [ ]:
unused_trucks = trucks_df[~trucks_df.truck_id.isin(trips_df.truck_id)]
display(unused_trucks.status.value_counts())

display(unused_trucks[unused_trucks.status == "Active"])

trucks_df["used"] = trucks_df.truck_id.isin(trips_df.truck_id)
display(trucks_df.groupby("status").used.value_counts().unstack())
trucks_df = trucks_df.drop(columns=["used"])

*Data Quality Note: Of the 120 trucks, 28 never appear in `trips`: 15 with status 
`Maintenance` and 13 with status `Inactive`. This split is exact, all 92 `Active` trucks 
have trips, and none of the `Maintenance`/`Inactive` trucks do. Truck status alone 
perfectly predicts trip presence in this dataset.*

### 2.3 Are there columns without variation?

In [ ]:
display(trucks_df.nunique())
display(trailers_df.nunique())

*Data Quality Note: `fuel_type` (trucks) and `status`/ `length_feet` (trailers) are
completely constant, every truck runs on the same fuel type, every trailer has the
same status and length. These columns carry no analytical value for group comparisons.
`trailer_number` has only 176 unique values across 180 trailers, meaning 4 numbers are
duplicated, a minor data quality issue worth noting, though not investigated further
here.*

### 2.4 Does the data match the dataset's own description?

In [ ]:
len(loads_df), len(delivery_events_df)

In [ ]:
delivery_events_df.on_time_flag.mean()

*Data Quality Note: actual row counts exceed the Kaggle description in both tables
checked (loads: 85,410 vs. "57,000+"; delivery_events: 170,820 vs. "114,000+"),
consistent with the discrepancy already noted in notebook 1. The overall on-time rate
(55.67%, combining pickup and delivery events) is also well below the claimed 85-95%
range, and matches almost exactly the average of the pickup (66.7%) and delivery
(44.6%) rates found in the earlier exploratory check, a good internal consistency
signal even though it contradicts the dataset's documentation. Both figures should be
kept in mind when citing the dataset's own claims elsewhere in this project.*